In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ExponentialLR

In [3]:
from torch.nn.utils.rnn import pad_sequence

In [4]:
from torch.utils.tensorboard import SummaryWriter

In [5]:
import numpy as np
import pandas as pd
import ast

In [6]:
import json

In [85]:
class CustomDataset(Dataset):
    def __init__(self, mode):
        self.mode = mode
        if mode == 'train':
            with open('data/ScanRefer_filtered_train_with_id.json', 'r') as file:
                data = json.load(file)
            self.labels = pd.read_csv('data/data_train.csv')['class'].to_numpy()
            self.nel_labels = pd.read_csv('data/data_train.csv')['nel'].to_numpy()
        else:
            with open('data/ScanRefer_filtered_val_with_id.json', 'r') as file:
                data = json.load(file)
            self.labels = pd.read_csv('data/data_val.csv')['class'].to_numpy()
            self.nel_labels = pd.read_csv('data/data_val.csv')['nel'].to_numpy()
        self.num_samples = len(data)
        
        # Simulating input data (features) as random floats
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        if self.mode == 'train':
            loaded_tensor = torch.load('data/contextual_data_train/{}.pt'.format(idx))[0]
        else:
            loaded_tensor = torch.load('data/contextual_data_val/{}.pt'.format(idx))[0]
        label = self.labels[idx]
        nel_label = ast.literal_eval(self.nel_labels[idx])
        nel_label_list = np.zeros(18)
        for i in nel_label:
            nel_label_list[i] = 1
        return loaded_tensor, label, nel_label_list.tolist()
        
def custom_collate_fn(batch):
    # Unpack batch into separate lists of inputs and labels
    inputs, labels, nel_labels = zip(*batch)

    # inputs: tuple of [L_i, 768] → pad to [B, max_len, 768]
    padded_inputs = pad_sequence(inputs, batch_first=True)  # Pads on dim=0 (sequence length)

    # Collect lengths (useful for masking or packing)
    lengths = torch.tensor([x.size(0) for x in inputs], dtype=torch.long)

    # Convert labels to tensor
    labels = torch.tensor(labels, dtype=torch.long)
    nel_labels = torch.tensor(nel_labels, dtype=torch.float)
    # nel_labels = torch.from_numpy(np.array(nel_labels)).float
    

    return padded_inputs, labels, nel_labels, lengths

In [86]:
class Classifier(nn.Module):
    def __init__(self, num_classes=18):
        super(Classifier, self).__init__()

        self.gru = nn.GRU(
            input_size=1024,
            hidden_size=512,
            batch_first=True,
            bidirectional=True
        )
        self.audio_classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
            )
        self.nel_classifier = nn.Sequential(
            nn.Linear(1024, 512),
            # nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            # nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
            )
    
    def forward(self, padded_tensor, lengths):
        batch_size = padded_tensor.shape[0]
        gru_out = []
        for i in range(batch_size):
            input_tensor = padded_tensor[i]
            input_tensor = input_tensor[:lengths[i], :].unsqueeze(0)
            _, hidden = self.gru(input_tensor)
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
            gru_out.append(hidden)
        x = torch.stack(gru_out).squeeze(1)
        c_x = self.audio_classifier(x)
        c_y = self.nel_classifier(x)
        # c_y = torch.sigmoid(c_y) ?
        return c_x, c_y  #

In [87]:
# Instantiate the dataset
train_dataset = CustomDataset(mode='train')
val_dataset = CustomDataset(mode='val')
# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=8, collate_fn=custom_collate_fn)

In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [89]:
model = Classifier().to(device)
# checkpoint = torch.load('model_state.pt')  # or 'model.pt', depending on your file
# model.load_state_dict(checkpoint)

In [90]:
loss_fn1 = nn.CrossEntropyLoss().to(device)
loss_fn2 = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ExponentialLR(optimizer, gamma=0.9)
epochs = 100

In [91]:
iter = 0
writer = SummaryWriter(log_dir='runs/classifier_experiment')
for epoch in range(epochs):
    model.train()
    print('training at epoch: {}'.format(epoch))
    scheduler.step()
    for padded_tensor, labels, nel_labels, lengths in train_loader:
        iter += 1
        padded_tensor = padded_tensor.to(device)
        labels = labels.to(device)
        nel_labels = nel_labels.to(device)
        lengths = lengths.to(device)
        outputs_1, outputs_2 = model(padded_tensor, lengths)
        # print(outputs.requires_grad)
        
        optimizer.zero_grad()
        loss_1 = loss_fn1(outputs_1, labels)
        loss_2 = loss_fn2(outputs_2, nel_labels)
        loss = loss_1 + loss_2
        loss.backward()
        optimizer.step()
        if iter % 10 == 0:
            writer.add_scalar('Loss1/train', loss_1, iter)
            writer.add_scalar('Loss2/train', loss_2, iter)
            print('loss 1: ', loss_1)
            print('loss 2: ', loss_2)

    model.eval()
    print('eval ...')
    correct = 0
    total = 0
    total_precision = 0
    total_recall = 0
    total_f1 = 0
    
    for padded_tensor, labels, nel_labels, lengths in val_loader:
        padded_tensor = padded_tensor.to(device)
        labels = labels.to(device)
        nel_labels = nel_labels.to(device)
        lengths = lengths.to(device)
        outputs_1, outputs_2 = model(padded_tensor, lengths)
        
        preds = outputs_1.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        probs = torch.sigmoid(outputs_2)
        preds = (probs > 0.5).int()
        nel_labels = nel_labels.int()
        TP = (preds & nel_labels).sum(dim=0)
        FP = (preds & (1 - nel_labels)).sum(dim=0)
        FN = ((1 - preds) & nel_labels).sum(dim=0)
        
        # Add epsilon to avoid division by zero
        eps = 1e-8
        
        precision = TP / (TP + FP + eps)
        recall    = TP / (TP + FN + eps)
        f1        = 2 * precision * recall / (precision + recall + eps)

        total_precision += precision
        total_recall += recall
        total_f1 += f1
    accuracy = correct / total
    total_precision = total_precision / total
    total_recall = total_recall / total
    total_f1 = total_f1 / total
    print('accuracy: ', accuracy)
    print('total_precision: ', total_precision)
    print('total_recall: ', total_recall)
    print('total_f1: ', total_f1)

    print('mean_precision: ', torch.mean(total_precision))
    print('mean_recall: ', torch.mean(total_recall))
    print('mean_f1: ', torch.mean(total_f1))

training at epoch: 0


/home/duccd/miniconda3/envs/test/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:131: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


loss 1:  tensor(2.7506, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.6219, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(2.6786, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.5084, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(2.5550, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.3801, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(2.5110, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2704, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(2.4322, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2572, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(2.5321, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2796, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss

KeyboardInterrupt: 